In [ ]:
import pandas as pd
import numpy as np

CSV_PATH    = '/content/merged_vicon_vive.csv'
OUTPUT_PATH = '/content/transformation_matrix_global_.txt'

VIVE_POS = ['controller_righthand_position_x',
            'controller_righthand_position_y',
            'controller_righthand_position_z']
VP_PREFIX = 'rightpoints'

df = pd.read_csv(CSV_PATH)

vicon_centroids = []
for _, row in df.iterrows():
    m1 = np.array([row[f'{VP_PREFIX}1_x'], row[f'{VP_PREFIX}1_y'], row[f'{VP_PREFIX}1_z']])
    m2 = np.array([row[f'{VP_PREFIX}2_x'], row[f'{VP_PREFIX}2_y'], row[f'{VP_PREFIX}2_z']])
    m3 = np.array([row[f'{VP_PREFIX}3_x'], row[f'{VP_PREFIX}3_y'], row[f'{VP_PREFIX}3_z']])
    centroid = (m1 + m2 + m3) / 3.0
    # centroid = m1
    vicon_centroids.append(centroid)
Q = np.vstack(vicon_centroids)

P = df[VIVE_POS].to_numpy()

centroid_P = P.mean(axis=0)
centroid_Q = Q.mean(axis=0)

P0 = P - centroid_P
Q0 = Q - centroid_Q

H = P0.T @ Q0

U, S, Vt = np.linalg.svd(H)

R_opt = Vt.T @ U.T
if np.linalg.det(R_opt) < 0:
    Vt[-1, :] *= -1
    R_opt = Vt.T @ U.T

t_opt = centroid_Q - R_opt @ centroid_P

T = np.eye(4)
T[:3, :3] = R_opt
T[:3, 3]  = t_opt

# with open(OUTPUT_PATH, 'w') as f:
#     for row in T:
#         f.write('[ ' + ' , '.join(f'{val:.6f}' for val in row) + ' ]\n')

# print("Computed global Vicon→Vive transformation:")
# print(T)
# print(f"Saved to: {OUTPUT_PATH}")

print(T)



[[ 0.6292178  -0.64570664 -0.43260594  2.21457465]
 [ 0.61071103  0.06646771  0.78905899 -0.23986469]
 [-0.4807463  -0.76068718  0.43616283  2.18952958]
 [ 0.          0.          0.          1.        ]]


================================================================================

In [ ]:
import pandas as pd
import numpy as np
from scipy.spatial.transform import Rotation as R

CSV_PATH   = '/content/merged_vicon_vive.csv'
STATIC_IDS = [25,26,27,28,29]

df = pd.read_csv(CSV_PATH)

local_estimates = []

for idx in STATIC_IDS:
    row = df.iloc[idx]
    # Vive pose
    p_v = np.array([row['controller_righthand_position_x'],
                    row['controller_righthand_position_y'],
                    row['controller_righthand_position_z']])
    q_v = np.array([row['controller_righthand_rotation_x'],
                    row['controller_righthand_rotation_y'],
                    row['controller_righthand_rotation_z'],
                    row['controller_righthand_rotation_w']])
    R_v = R.from_quat(q_v).as_matrix()
    Rv_inv = R_v.T

    # Vicon markers
    m1 = np.array([row['rightpoints1_x'], row['rightpoints1_y'], row['rightpoints1_z']])
    m2 = np.array([row['rightpoints2_x'], row['rightpoints2_y'], row['rightpoints2_z']])
    m3 = np.array([row['rightpoints3_x'], row['rightpoints3_y'], row['rightpoints3_z']])

    # Compute local offsets
    ell1 = Rv_inv.dot(m1 - p_v)
    ell2 = Rv_inv.dot(m2 - p_v)
    ell3 = Rv_inv.dot(m3 - p_v)

    local_estimates.append(np.vstack([ell1, ell2, ell3]))

local_estimates = np.stack(local_estimates)

local_markers = local_estimates.mean(axis=0)

print("Local marker offsets:")
for i, ell in enumerate(local_markers, start=1):
    print(f"  Marker {i}: {ell}")


Local marker offsets (in controller frame):
  Marker 1: [ 1.83019209 -0.78199164 -1.65414877]
  Marker 2: [ 1.77237123 -0.78847157 -1.67046533]
  Marker 3: [ 1.86662454 -0.86314124 -1.63416716]


In [ ]:
import pandas as pd
import numpy as np
from scipy.spatial.transform import Rotation as R

CSV_PATH    = '/content/merged_vicon_vive.csv'
# OUTPUT_PATH = 'transformation_matrix_global.txt'

# Vive pose columns
VIVE_POS = ['controller_righthand_position_x',
            'controller_righthand_position_y',
            'controller_righthand_position_z']
VIVE_ROT = ['controller_righthand_rotation_x',
            'controller_righthand_rotation_y',
            'controller_righthand_rotation_z',
            'controller_righthand_rotation_w']

# Vicon marker columns
MARKER_PREFIX = 'rightpoints'

local_markers = np.array([
    [ 1.83019209, -0.78199164, -1.65414877],
    [ 1.77237123, -0.78847157, -1.67046533],
    [ 1.86662454, -0.86314124, -1.63416716]
])

df = pd.read_csv(CSV_PATH)
P_pos = df[VIVE_POS].to_numpy()
P_rot = df[VIVE_ROT].to_numpy()
M1 = df[[f'{MARKER_PREFIX}1_x', f'{MARKER_PREFIX}1_y', f'{MARKER_PREFIX}1_z']].to_numpy()
M2 = df[[f'{MARKER_PREFIX}2_x', f'{MARKER_PREFIX}2_y', f'{MARKER_PREFIX}2_z']].to_numpy()
M3 = df[[f'{MARKER_PREFIX}3_x', f'{MARKER_PREFIX}3_y', f'{MARKER_PREFIX}3_z']].to_numpy()

cloud_vive = []
for pos, quat in zip(P_pos, P_rot):
    Rv = R.from_quat(quat).as_matrix()
    for lm in local_markers:
        cloud_vive.append(Rv.dot(lm) + pos)
cloud_vive = np.vstack(cloud_vive)

cloud_vicon = np.vstack((M1, M2, M3))

cV = cloud_vive.mean(axis=0)
cC = cloud_vicon.mean(axis=0)

P0 = cloud_vive - cV
Q0 = cloud_vicon - cC

H = P0.T @ Q0
U, _, Vt = np.linalg.svd(H)
R_opt = Vt.T @ U.T
if np.linalg.det(R_opt) < 0:
    Vt[-1,:] *= -1
    R_opt = Vt.T @ U.T

t_opt = cC - R_opt @ cV

T = np.eye(4)
T[:3,:3] = R_opt
T[:3, 3] = t_opt

print(T)
# with open(OUTPUT_PATH, 'w') as f:
#     for row in T:
#         f.write('[ ' + ' , '.join(f'{v:.6f}' for v in row) + ' ]\n')

# print("Saved global transform to", OUTPUT_PATH)
# print(T)


[[ 0.68865494  0.58102426 -0.4337801   0.34250803]
 [ 0.69121037 -0.34531617  0.63481097 -1.96858607]
 [ 0.21904929 -0.73699902 -0.63941368  0.97818181]
 [ 0.          0.          0.          1.        ]]
